---
## 📦 Step 1: Install Required Libraries

In [5]:
# Install all required packages
!pip install langchain langchain-core langchain-groq langsmith python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.3 MB/s eta 0:00:00



## 🔑 Step 2: Configure API Keys & Enable LangSmith Tracing


In [2]:
import os

# ─── LangSmith Tracing Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "AI-Resume-Screening"
os.environ["LANGCHAIN_API_KEY"]    = "Enter your API-KEY"

# ─── Groq API Key
os.environ["GROQ_API_KEY"]         = "Enter your API-KEY"

print("✅ Environment variables set!")
print(f"   Tracing enabled : {os.environ['LANGCHAIN_TRACING_V2']}")
print(f"   Project         : {os.environ['LANGCHAIN_PROJECT']}")

✅ Environment variables set!
   Tracing enabled : true
   Project         : AI-Resume-Screening



## 🛠️ Step 3: LLM Utility

In [7]:
# utils/config.py — LLM factory function
from langchain_groq import ChatGroq

def get_llm(temperature: float = 0.0) -> ChatGroq:

    return ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=temperature
    )

# Quick sanity check
llm_test = get_llm()
response = llm_test.invoke("Say 'LLM Ready' in exactly 2 words.")
print(f"✅ LLM check: {response.content.strip()}")

✅ LLM check: LLM Ready


---
## 📄 Step 4: Sample Data — 3 Resumes + 1 Job Description


In [8]:
# ─── Job Description ────────────────────────────────────────────────────────
JOB_DESCRIPTION = """
Position: Data Scientist
Company: TechCorp Analytics

Required Skills:
- Python (5+ years)
- Machine Learning (scikit-learn, XGBoost, LightGBM)
- Deep Learning (TensorFlow or PyTorch)
- SQL and data wrangling (pandas, numpy)
- Statistical modeling and hypothesis testing
- Data visualization (Matplotlib, Seaborn, Plotly)
- Cloud platforms (AWS, GCP, or Azure)
- MLOps tools (MLflow, Docker, Kubernetes)
- Strong communication skills

Experience Required: 3–5 years in a data science role
Education: Bachelor's or Master's in CS, Statistics, or related field
"""

# ─── Strong Candidate ────────────────────────────────────────────────────────
RESUME_STRONG = """
Name: Priya Sharma
Experience: 4 years as Data Scientist at Flipkart AI Labs
Education: M.Tech in Computer Science, IIT Bombay

Skills:
- Python (6 years), SQL, R
- Machine Learning: scikit-learn, XGBoost, LightGBM, CatBoost
- Deep Learning: TensorFlow, PyTorch, Keras
- Data tools: pandas, numpy, Spark
- Cloud: AWS (SageMaker, S3, EC2), GCP (BigQuery)
- MLOps: MLflow, Docker, Kubernetes, Airflow
- Visualization: Matplotlib, Seaborn, Plotly, Tableau
- Statistics: A/B testing, hypothesis testing, Bayesian methods

Projects:
- Built real-time recommendation engine serving 10M+ users (XGBoost + Kafka)
- Deployed NLP fraud detection model reducing fraud by 35% (PyTorch + AWS)
- Led 3-member team on demand forecasting pipeline (LightGBM + MLflow + Docker)

Certifications: AWS Certified ML Specialty, Google Professional Data Engineer
"""

# ─── Average Candidate ───────────────────────────────────────────────────────
RESUME_AVERAGE = """
Name: Rahul Verma
Experience: 2.5 years as Junior Data Analyst at Infosys
Education: B.Tech in Information Technology, VIT University

Skills:
- Python (3 years), SQL
- Machine Learning: scikit-learn, basic regression/classification
- Data tools: pandas, numpy, Excel
- Visualization: Matplotlib, Power BI
- Some experience with AWS S3 and EC2

Projects:
- Customer churn prediction using logistic regression (scikit-learn)
- Sales dashboard built in Power BI for quarterly reports
- Data cleaning pipeline for e-commerce dataset (Python + pandas)

Certifications: IBM Data Science Professional Certificate
"""

# ─── Weak Candidate ──────────────────────────────────────────────────────────
RESUME_WEAK = """
Name: Anil Kumar
Experience: 6 months internship as Web Developer at a startup
Education: B.Com, Delhi University

Skills:
- HTML, CSS, JavaScript (basic)
- Some Python (beginner level — completed a 30-day Python bootcamp)
- MS Office (Word, Excel, PowerPoint)
- Basic understanding of databases (MySQL intro course)

Projects:
- Built a personal portfolio website using HTML/CSS
- Created a simple calculator app in Python as part of bootcamp
- Managed social media content for the startup
"""

print("✅ Resume data loaded:")
print(f"   Strong  : Priya Sharma (4 yrs, IIT Bombay M.Tech)")
print(f"   Average : Rahul Verma (2.5 yrs, VIT B.Tech)")
print(f"   Weak    : Anil Kumar (6 months intern, B.Com)")

✅ Resume data loaded:
   Strong  : Priya Sharma (4 yrs, IIT Bombay M.Tech)
   Average : Rahul Verma (2.5 yrs, VIT B.Tech)
   Weak    : Anil Kumar (6 months intern, B.Com)



## 📝 Step 5: Prompt Templates


In [9]:
# prompts/extraction_prompt.py
from langchain_core.prompts import PromptTemplate

extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are a professional resume parser. Extract information ONLY from the resume text provided.
Do NOT assume, infer, or add any skills, tools, or experience NOT explicitly mentioned.

Resume:
{resume}

Extract and return EXACTLY the following JSON structure (no markdown, no extra text):
{{
  "name": "candidate name",
  "years_experience": <number or 0 if unclear>,
  "education": "degree and institution",
  "skills": ["skill1", "skill2", ...],
  "tools": ["tool1", "tool2", ...],
  "key_projects": ["project1 summary", "project2 summary", ...]
}}

Return ONLY valid JSON. No commentary.
"""
)

print("✅ extraction_prompt loaded")

✅ extraction_prompt loaded


In [10]:
# prompts/matching_prompt.py
matching_prompt = PromptTemplate(
    input_variables=["extracted_profile", "job_description"],
    template="""
You are a senior recruiter performing a skill-gap analysis.

Job Description:
{job_description}

Candidate Profile (extracted from resume):
{extracted_profile}

Instructions:
- Compare the candidate's skills and experience ONLY against the job requirements.
- Do NOT award credit for skills not listed in the candidate profile.
- Be objective and precise.

Return EXACTLY this JSON (no markdown, no extra text):
{{
  "matched_skills": ["skill1", "skill2", ...],
  "missing_skills": ["skill1", "skill2", ...],
  "experience_match": "strong | partial | weak",
  "education_match": "strong | partial | weak",
  "overall_match": "strong | partial | weak"
}}

Return ONLY valid JSON.
"""
)

print("✅ matching_prompt loaded")

✅ matching_prompt loaded


In [11]:
# prompts/scoring_prompt.py
scoring_prompt = PromptTemplate(
    input_variables=["extracted_profile", "match_analysis", "job_description"],
    template="""
You are an AI hiring system that assigns a numerical fit score.

Job Description:
{job_description}

Candidate Profile:
{extracted_profile}

Match Analysis:
{match_analysis}

Scoring Rubric:
- Skills match      : 40 points max (proportion of required skills present)
- Experience match  : 25 points max (years and relevance)
- Education match   : 15 points max (degree and field relevance)
- Tools & MLOps     : 20 points max (cloud, DevOps, MLOps tools)
Total: 100 points

Instructions:
- Assign scores based ONLY on what is present in the candidate profile.
- Do NOT award points for skills or experience not explicitly mentioned.
- Provide a clear breakdown and a final score.

Return EXACTLY this JSON (no markdown, no extra text):
{{
  "skills_score": <0-40>,
  "experience_score": <0-25>,
  "education_score": <0-15>,
  "tools_score": <0-20>,
  "total_score": <0-100>,
  "grade": "A (85-100) | B (70-84) | C (50-69) | D (below 50)",
  "recommendation": "Strongly Recommend | Recommend | Consider | Reject",
  "explanation": "2-3 sentence explanation of why this score was assigned, citing specific evidence from the resume"
}}

Return ONLY valid JSON.
"""
)

print("✅ scoring_prompt loaded")

✅ scoring_prompt loaded



## ⛓️ Step 6: LangChain LCEL Chains (chains/ module)


In [12]:
# chains/extraction_chain.py
# Low temperature = deterministic extraction (no hallucination)
extraction_llm = get_llm(temperature=0.0)
extraction_chain = extraction_prompt | extraction_llm

# chains/matching_chain.py
# Low temperature = objective comparison
matching_llm = get_llm(temperature=0.0)
matching_chain = matching_prompt | matching_llm

# chains/scoring_chain.py
# Slightly higher temperature for nuanced explanation
scoring_llm = get_llm(temperature=0.1)
scoring_chain = scoring_prompt | scoring_llm

print("✅ All LCEL chains created:")
print("   extraction_chain  = extraction_prompt | llm(temp=0.0)")
print("   matching_chain    = matching_prompt   | llm(temp=0.0)")
print("   scoring_chain     = scoring_prompt    | llm(temp=0.1)")

✅ All LCEL chains created:
   extraction_chain  = extraction_prompt | llm(temp=0.0)
   matching_chain    = matching_prompt   | llm(temp=0.0)
   scoring_chain     = scoring_prompt    | llm(temp=0.1)



## 🚀 Step 7: Main Pipeline (main.py equivalent)


In [15]:
import json

def safe_parse_json(text: str) -> dict:

    # Strip markdown fences if present
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        cleaned = "\n".join(lines[1:-1])  # remove first and last fence lines
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        return {"parse_error": str(e), "raw": text[:300]}


def screen_resume(resume: str, job_description: str, candidate_type: str) -> dict:

    print(f"\n{'='*60}")
    print(f"📋 Screening: {candidate_type} Candidate")
    print(f"{'='*60}")

    # ── Step 1: Skill Extraction ─────────────────────────────────
    print("\n🔍 Step 1: Extracting skills and profile...")
    extraction_result = extraction_chain.invoke({"resume": resume})
    extracted_profile = extraction_result.content.strip()
    profile_data = safe_parse_json(extracted_profile)
    print(f"   Name        : {profile_data.get('name', 'N/A')}")
    print(f"   Experience  : {profile_data.get('years_experience', 'N/A')} years")
    print(f"   Education   : {profile_data.get('education', 'N/A')}")
    print(f"   Skills count: {len(profile_data.get('skills', []))}")

    # ── Step 2: Matching Logic ───────────────────────────────────
    print("\n🔗 Step 2: Matching profile against job requirements...")
    matching_result = matching_chain.invoke({
        "extracted_profile": extracted_profile,
        "job_description": job_description
    })
    match_analysis = matching_result.content.strip()
    match_data = safe_parse_json(match_analysis)
    print(f"   Matched skills : {len(match_data.get('matched_skills', []))}")
    print(f"   Missing skills : {len(match_data.get('missing_skills', []))}")
    print(f"   Overall match  : {match_data.get('overall_match', 'N/A')}")

    # ── Step 3: Scoring + Explanation ───────────────────────────
    print("\n📊 Step 3: Calculating fit score and explanation...")
    scoring_result = scoring_chain.invoke({
        "extracted_profile": extracted_profile,
        "match_analysis": match_analysis,
        "job_description": job_description
    })
    score_text = scoring_result.content.strip()
    score_data = safe_parse_json(score_text)
    print(f"   Total Score    : {score_data.get('total_score', 'N/A')}/100")
    print(f"   Grade          : {score_data.get('grade', 'N/A')}")
    print(f"   Recommendation : {score_data.get('recommendation', 'N/A')}")

    return {
        "candidate_type" : candidate_type,
        "profile"        : profile_data,
        "match_analysis" : match_data,
        "scoring"        : score_data
    }

print("✅ screen_resume() pipeline function defined")

✅ screen_resume() pipeline function defined



## 🏃 Step 8: Run 3 Screening Sessions (Strong, Average, Weak)


In [16]:
# ─── Run 1: Strong Candidate ─────────────────────────────────────────────────
result_strong = screen_resume(
    resume=RESUME_STRONG,
    job_description=JOB_DESCRIPTION,
    candidate_type="STRONG"
)


📋 Screening: STRONG Candidate

🔍 Step 1: Extracting skills and profile...
   Name        : Priya Sharma
   Experience  : 4 years
   Education   : M.Tech in Computer Science, IIT Bombay
   Skills count: 6

🔗 Step 2: Matching profile against job requirements...
   Matched skills : 8
   Missing skills : 4
   Overall match  : partial

📊 Step 3: Calculating fit score and explanation...
   Total Score    : 77/100
   Grade          : B (70-84)
   Recommendation : Recommend


In [17]:
# ─── Run 2: Average Candidate ─────────────────────────────────────────────────
result_average = screen_resume(
    resume=RESUME_AVERAGE,
    job_description=JOB_DESCRIPTION,
    candidate_type="AVERAGE"
)


📋 Screening: AVERAGE Candidate

🔍 Step 1: Extracting skills and profile...
   Name        : Rahul Verma
   Experience  : 2.5 years
   Education   : B.Tech in Information Technology, VIT University
   Skills count: 4

🔗 Step 2: Matching profile against job requirements...
   Matched skills : 7
   Missing skills : 12
   Overall match  : weak

📊 Step 3: Calculating fit score and explanation...
   Total Score    : 51.5/100
   Grade          : C (50-69)
   Recommendation : Consider


In [18]:
# ─── Run 3: Weak Candidate ────────────────────────────────────────────────────
result_weak = screen_resume(
    resume=RESUME_WEAK,
    job_description=JOB_DESCRIPTION,
    candidate_type="WEAK"
)


📋 Screening: WEAK Candidate

🔍 Step 1: Extracting skills and profile...
   Name        : Anil Kumar
   Experience  : 0 years
   Education   : B.Com, Delhi University
   Skills count: 6

🔗 Step 2: Matching profile against job requirements...
   Matched skills : 1
   Missing skills : 8
   Overall match  : weak

📊 Step 3: Calculating fit score and explanation...
   Total Score    : 5/100
   Grade          : D (below 50)
   Recommendation : Reject



## 📊 Step 9: Results Summary & Comparison

In [19]:
print("\n" + "="*70)
print("                  📊 RESUME SCREENING RESULTS SUMMARY")
print("="*70)
print(f"{'Candidate':<12} {'Name':<18} {'Score':>7} {'Grade':>7} {'Recommendation':<25}")
print("-"*70)

for result in [result_strong, result_average, result_weak]:
    name     = result["profile"].get("name", "N/A")
    score    = result["scoring"].get("total_score", "N/A")
    grade    = result["scoring"].get("grade", "N/A")
    rec      = result["scoring"].get("recommendation", "N/A")
    ctype    = result["candidate_type"]
    print(f"{ctype:<12} {name:<18} {str(score):>7} {str(grade):>7} {rec:<25}")

print("="*70)

print("\n📋 Score Breakdown:")
print(f"{'Candidate':<12} {'Skills':>8} {'Exp':>6} {'Edu':>6} {'Tools':>7} {'Total':>7}")
print("-"*50)
for result in [result_strong, result_average, result_weak]:
    sc    = result["scoring"]
    ctype = result["candidate_type"]
    print(f"{ctype:<12} {str(sc.get('skills_score','?')):>8} "
          f"{str(sc.get('experience_score','?')):>6} "
          f"{str(sc.get('education_score','?')):>6} "
          f"{str(sc.get('tools_score','?')):>7} "
          f"{str(sc.get('total_score','?')):>7}")


                  📊 RESUME SCREENING RESULTS SUMMARY
Candidate    Name                 Score   Grade Recommendation           
----------------------------------------------------------------------
STRONG       Priya Sharma            77 B (70-84) Recommend                
AVERAGE      Rahul Verma           51.5 C (50-69) Consider                 
WEAK         Anil Kumar               5 D (below 50) Reject                   

📋 Score Breakdown:
Candidate      Skills    Exp    Edu   Tools   Total
--------------------------------------------------
STRONG             24     20     15      18      77
AVERAGE            16   12.5     15       8    51.5
WEAK                5      0      0       0       5



## 💬 Step 10: Detailed Explanations (Explainability)

In [20]:
for result in [result_strong, result_average, result_weak]:
    name        = result["profile"].get("name", "N/A")
    ctype       = result["candidate_type"]
    explanation = result["scoring"].get("explanation", "No explanation generated.")
    matched     = result["match_analysis"].get("matched_skills", [])
    missing     = result["match_analysis"].get("missing_skills", [])

    print(f"\n{'─'*60}")
    print(f"👤 {name} ({ctype} Candidate)")
    print(f"{'─'*60}")
    print(f"📌 Explanation : {explanation}")
    print(f"✅ Matched     : {', '.join(matched[:5])}{'...' if len(matched) > 5 else ''}")
    print(f"❌ Missing     : {', '.join(missing[:5])}{'...' if len(missing) > 5 else ''}")


────────────────────────────────────────────────────────────
👤 Priya Sharma (STRONG Candidate)
────────────────────────────────────────────────────────────
📌 Explanation : Priya Sharma's profile shows a strong match in experience and education, with 4 years of experience and a Master's in Computer Science. The skills section of the profile mentions relevant skills like Python, Machine Learning, and Deep Learning, but lacks explicit mentions of some required skills like data wrangling with pandas and numpy. The tools and MLOps section is well-covered with mentions of scikit-learn, TensorFlow, PyTorch, and cloud platforms.
✅ Matched     : Python, Machine Learning, Deep Learning, SQL, Statistical modeling and hypothesis testing...
❌ Missing     : data wrangling (pandas, numpy) explicitly mentioned, XGBoost and LightGBM explicitly mentioned under Machine Learning, TensorFlow or PyTorch explicitly mentioned under Deep Learning, Strong communication skills

─────────────────────────────────


## 🐛 Step 11: LangSmith Debugging — Demonstrating Incorrect Output


In [21]:
# 🐛 Debug Case: Empty / malformed resume
# This demonstrates LangSmith catching edge-case pipeline behavior

RESUME_EDGE_CASE = """
Name: Unknown Person
I am a quick learner and hardworking individual.
I know computers and internet.
I want to work in your company.
"""

print("🐛 Running edge case to demonstrate LangSmith debugging...")
print("   Input: Vague resume with no specific skills or experience")
print()

# Extract only — to show incorrect/minimal output captured in LangSmith
edge_extraction = extraction_chain.invoke({"resume": RESUME_EDGE_CASE})
edge_data = safe_parse_json(edge_extraction.content)

print("📊 Extraction result (edge case):")
print(json.dumps(edge_data, indent=2))

print()
print("💡 LangSmith Debugging Insight:")
print("   - The model correctly returns 0 years_experience and empty skills list")
print("   - This is the CORRECT behavior — no hallucination of skills")
print("   - LangSmith trace shows exactly what prompt was sent and what was returned")
print("   - Useful for identifying prompts that produce unexpected outputs")
print("   ✅ View trace at: https://smith.langchain.com/ → Project: AI-Resume-Screening")

🐛 Running edge case to demonstrate LangSmith debugging...
   Input: Vague resume with no specific skills or experience

📊 Extraction result (edge case):
{
  "name": "Unknown Person",
  "years_experience": 0,
  "education": "",
  "skills": [
    "quick learner",
    "hardworking"
  ],
  "tools": [
    "computers",
    "internet"
  ],
  "key_projects": []
}

💡 LangSmith Debugging Insight:
   - The model correctly returns 0 years_experience and empty skills list
   - This is the CORRECT behavior — no hallucination of skills
   - LangSmith trace shows exactly what prompt was sent and what was returned
   - Useful for identifying prompts that produce unexpected outputs
   ✅ View trace at: https://smith.langchain.com/ → Project: AI-Resume-Screening



## 🏷️ Step 12: Bonus — LangSmith Tags & Structured JSON Output

In [22]:
# Bonus: Using LangSmith tags for better trace organization
from langchain_core.runnables import RunnableConfig

def screen_resume_with_tags(resume: str, job_description: str,
                             candidate_type: str, tags: list) -> dict:
    """
    Enhanced screening with LangSmith tags for trace organization.
    Tags appear in LangSmith dashboard for easy filtering.
    """
    config = RunnableConfig(tags=tags, metadata={"candidate_type": candidate_type})

    extraction_result = extraction_chain.invoke(
        {"resume": resume}, config=config
    )
    extracted_profile = extraction_result.content.strip()

    matching_result = matching_chain.invoke(
        {"extracted_profile": extracted_profile, "job_description": job_description},
        config=config
    )

    scoring_result = scoring_chain.invoke(
        {
            "extracted_profile": extracted_profile,
            "match_analysis"   : matching_result.content.strip(),
            "job_description"  : job_description
        },
        config=config
    )

    score_data = safe_parse_json(scoring_result.content)
    return {
        "candidate_type" : candidate_type,
        "total_score"    : score_data.get("total_score"),
        "recommendation" : score_data.get("recommendation"),
        "langsmith_tags" : tags
    }


print("🏷️  Running tagged LangSmith traces (Bonus feature)...")

tagged_result = screen_resume_with_tags(
    resume=RESUME_STRONG,
    job_description=JOB_DESCRIPTION,
    candidate_type="STRONG",
    tags=["resume-screening", "strong-candidate", "data-scientist-role"]
)

print(f"✅ Tagged run complete:")
print(f"   Score          : {tagged_result['total_score']}/100")
print(f"   Recommendation : {tagged_result['recommendation']}")
print(f"   LangSmith tags : {tagged_result['langsmith_tags']}")
print()
print("📸 Check LangSmith dashboard — traces will show these tags for easy filtering!")

🏷️  Running tagged LangSmith traces (Bonus feature)...
✅ Tagged run complete:
   Score          : 87/100
   Recommendation : Strongly Recommend
   LangSmith tags : ['resume-screening', 'strong-candidate', 'data-scientist-role']

📸 Check LangSmith dashboard — traces will show these tags for easy filtering!



## 🎯 Step 13: Few-Shot Prompting (Bonus)


In [23]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate as PT

# Few-shot examples to guide the model toward consistent scoring
few_shot_examples = [
    {
        "profile_summary": "5 years Python, ML expert, AWS certified, MLflow used",
        "quick_score": "88/100 — Grade A — Strongly Recommend"
    },
    {
        "profile_summary": "2 years Python, basic scikit-learn, no cloud experience",
        "quick_score": "52/100 — Grade C — Consider"
    },
    {
        "profile_summary": "No ML skills, 6 months web dev internship, B.Com graduate",
        "quick_score": "12/100 — Grade D — Reject"
    }
]

example_template = PT(
    input_variables=["profile_summary", "quick_score"],
    template="Profile: {profile_summary}\nScore: {quick_score}"
)

few_shot_quick_score_prompt = FewShotPromptTemplate(
    examples=few_shot_examples,
    example_prompt=example_template,
    prefix="You are a resume scorer. Score the following candidate for a Data Scientist role.\nExamples:",
    suffix="\nProfile: {profile_summary}\nScore:",
    input_variables=["profile_summary"]
)

few_shot_chain = few_shot_quick_score_prompt | get_llm(temperature=0.0)

# Test with Average candidate summary
test_summary = "2.5 years Python and SQL, scikit-learn basics, Power BI, basic AWS, B.Tech IT"
few_shot_result = few_shot_chain.invoke({"profile_summary": test_summary})

print("🎯 Few-Shot Quick Score Demo:")
print(f"   Input  : {test_summary}")
print(f"   Output : {few_shot_result.content.strip()}")
print("\n✅ Few-shot prompting guides the model to produce consistent score format")

🎯 Few-Shot Quick Score Demo:
   Input  : 2.5 years Python and SQL, scikit-learn basics, Power BI, basic AWS, B.Tech IT
   Output : Profile: 2.5 years Python and SQL, scikit-learn basics, Power BI, basic AWS, B.Tech IT
Score: 68/100 — Grade B — Recommend

Reasoning: 
- The candidate has a decent amount of experience in Python and SQL, which are essential skills for a Data Scientist.
- They have basic knowledge of scikit-learn, which is a popular machine learning library in Python.
- Experience with Power BI is a plus, as it shows they have some data visualization skills.
- Basic AWS knowledge is also a positive, as cloud experience is valuable in the field of Data Science.
- The candidate's educational background in B.Tech IT is relevant to the role.
However, the candidate's overall experience and skill set are not as strong as those of an ideal candidate, which is why they score a B rather than an A.

✅ Few-shot prompting guides the model to produce consistent score format
